# Institutional Transaction Cost Analysis & Optimal Execution Research Platform

## Project Overview

Large institutional investors cannot execute large equity orders with a single trade without affecting market prices. The objective of this project is to analyze how different execution algorithms influence transaction costs, market impact, and execution risk under varying liquidity conditions.

This project develops a Python-based Transaction Cost Analysis (TCA) platform that combines market microstructure concepts with quantitative finance techniques to evaluate institutional trade execution. The analysis is based on intraday equity market data and incorporates liquidity estimation, market impact modeling, Monte Carlo simulation, and execution performance benchmarking.

## Project Objectives

The platform performs the following analyses:

- Downloads real 1-minute intraday equity market data from Yahoo Finance.
- Estimates market liquidity using Average Daily Volume (ADV), realized volatility, and the Corwin–Schultz bid-ask spread estimator.
- Calibrates the Almgren–Chriss temporary and permanent market impact model.
- Simulates and compares four institutional execution strategies:
  - Time Weighted Average Price (TWAP)
  - Volume Weighted Average Price (VWAP)
  - Percentage of Volume (POV)
  - Almgren–Chriss Implementation Shortfall Optimal Execution
- Evaluates execution quality using:
  - Implementation Shortfall
  - Slippage Distribution
  - Transaction Cost Decomposition
  - Execution Risk
- Visualizes execution performance through interactive Plotly dashboards and comparative analytics.

## Business Relevance

Transaction Cost Analysis is widely used by asset managers, investment banks, hedge funds, and electronic trading desks to evaluate execution quality and reduce trading costs. Although this project uses publicly available market data, the analytical workflow reflects concepts commonly applied in institutional execution research and quantitative trading.

## Data Source

This project primarily uses 1-minute intraday data obtained through Yahoo Finance.

Because Yahoo Finance does not provide historical order book or bid-ask quote data, bid-ask spreads are estimated using the Corwin–Schultz high-low spread estimator. If live market data is unavailable because of network issues, market holidays, or API limitations, the notebook automatically generates synthetic intraday data so that the complete analytical workflow can still be reproduced.

In [ ]:
# =============================================================================
# Environment Setup
# Institutional Transaction Cost Analysis & Optimal Execution Research Platform
# =============================================================================

# Install external libraries that are not included in the default
# Google Colab environment.

!pip -q install yfinance statsmodels kaleido

# =============================================================================
# Import Required Libraries
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from scipy import stats
from scipy.optimize import brentq
import statsmodels.api as sm

import yfinance as yf

# =============================================================================
# Display Configuration
# =============================================================================

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.random.seed(42)

print("=" * 70)
print("Environment initialized successfully")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print(f"yfinance   : {yf.__version__}")
print("=" * 70)

Environment initialized successfully
NumPy      : 2.0.2
Pandas     : 2.2.2
yfinance   : 0.2.66


## 1. Research Configuration

The configuration below defines the inputs used throughout the Transaction Cost Analysis framework. These parameters specify the security under analysis, parent order characteristics, execution model assumptions, and Monte Carlo simulation settings. Modifying the configuration allows the same analytical workflow to be applied to different execution scenarios while maintaining a consistent research methodology.

In [ ]:
# =============================================================================
# Research Configuration
# =============================================================================
# Update these values to rerun the analysis for a different ticker, order size,
# execution horizon, or simulation setting.
# =============================================================================

CONFIG = {
    # Market data
    "TICKER": "AAPL",
    "INTRADAY_PERIOD": "5d",
    "INTRADAY_INTERVAL": "1m",

    # Parent order
    "ORDER_SIDE": "BUY",          # BUY or SELL
    "ORDER_SIZE_SHARES": 250_000, # Total shares in the parent order
    "PARTICIPATION_TARGET": 0.10, # POV target participation rate
    "N_SLICES": 13,               # Number of execution slices across the day

    # Almgren-Chriss model
    "RISK_AVERSION": 3e-2,

    # Monte Carlo simulation
    "N_MC_PATHS": 2_000,
    "RANDOM_SEED": 42,
}

print("=" * 70)
print("Research Configuration")
print("=" * 70)

for key, value in CONFIG.items():
    print(f"{key:<24}: {value}")

print("=" * 70)

Research Configuration
TICKER                  : AAPL
INTRADAY_PERIOD         : 5d
INTRADAY_INTERVAL       : 1m
ORDER_SIDE              : BUY
ORDER_SIZE_SHARES       : 250000
PARTICIPATION_TARGET    : 0.1
N_SLICES                : 13
RISK_AVERSION           : 0.03
N_MC_PATHS              : 2000
RANDOM_SEED             : 42


## 2. Market Data

This study uses one-minute OHLCV (Open, High, Low, Close, Volume) data obtained through Yahoo Finance via `yfinance`. Although Yahoo Finance does not provide historical order book or bid-ask quote data, it offers sufficient intraday information to estimate market liquidity and support transaction cost analysis.

To ensure the analytical workflow remains reproducible, the notebook automatically generates synthetic intraday market data whenever live data cannot be retrieved because of API limitations, market holidays, or connectivity issues. The synthetic data follows realistic intraday volume and volatility patterns, allowing every stage of the analysis to be executed consistently.

In [ ]:
# =============================================================================
# Data Collection
# =============================================================================

def generate_synthetic_intraday(
    ticker: str = "SYNTH",
    n_days: int = 5,
    minutes_per_day: int = 390,
    start_price: float = 150.0,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Generate synthetic 1-minute OHLCV data for fallback execution.
    """
    rng = np.random.default_rng(seed)
    rows = []
    price = start_price
    sigma_t = 0.0006

    for day_idx in range(n_days):
        day_start = pd.Timestamp.today().normalize() - pd.Timedelta(days=(n_days - day_idx))
        day_start = day_start.replace(hour=9, minute=30)

        t = np.linspace(0, 1, minutes_per_day)
        volume_curve = 1.6 * np.exp(-t * 9) + 1.6 * np.exp(-(1 - t) * 9) + 0.25
        volume_curve = volume_curve / volume_curve.sum()

        day_total_volume = rng.integers(30_000_000, 70_000_000)

        for minute_idx in range(minutes_per_day):
            sigma_t = 0.90 * sigma_t + 0.10 * 0.0006 + rng.normal(0, 0.00003)
            sigma_t = max(sigma_t, 0.0001)

            ret = rng.normal(0, sigma_t)
            price *= (1 + ret)

            open_price = price * (1 + rng.normal(0, sigma_t * 0.2))
            high_price = max(open_price, price) * (1 + abs(rng.normal(0, sigma_t * 0.5)))
            low_price = min(open_price, price) * (1 - abs(rng.normal(0, sigma_t * 0.5)))
            volume = max(int(day_total_volume * volume_curve[minute_idx] * rng.uniform(0.4, 1.8)), 100)

            timestamp = day_start + pd.Timedelta(minutes=minute_idx)
            rows.append([timestamp, open_price, high_price, low_price, price, volume])

    df = pd.DataFrame(rows, columns=["Datetime", "Open", "High", "Low", "Close", "Volume"])
    df = df.set_index("Datetime")
    df.attrs["source"] = "synthetic"
    return df


def fetch_intraday_data(ticker: str, period: str, interval: str) -> pd.DataFrame:
    """
    Fetch intraday OHLCV data from Yahoo Finance.
    Falls back to synthetic data if live data is unavailable.
    """
    try:
        raw = yf.download(
            ticker,
            period=period,
            interval=interval,
            progress=False,
            auto_adjust=False,
            threads=False,
        )

        if raw is None or raw.empty:
            raise ValueError("Empty dataframe returned by yfinance.")

        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)

        raw = raw[["Open", "High", "Low", "Close", "Volume"]].dropna()

        if len(raw) < 100:
            raise ValueError(f"Only {len(raw)} bars returned; insufficient for intraday analysis.")

        raw.index.name = "Datetime"
        raw.attrs["source"] = "yfinance"
        return raw

    except Exception as exc:
        print(f"[WARN] Live data fetch failed ({exc!r}). Falling back to synthetic intraday data.")
        return generate_synthetic_intraday(
            ticker=ticker,
            n_days=5,
            seed=CONFIG["RANDOM_SEED"],
        )


intraday = fetch_intraday_data(
    CONFIG["TICKER"],
    CONFIG["INTRADAY_PERIOD"],
    CONFIG["INTRADAY_INTERVAL"],
)

print("=" * 70)
print(f"Data source      : {intraday.attrs.get('source', 'unknown')}")
print(f"Rows             : {len(intraday):,}")
print(f"Date range       : {intraday.index.min()} -> {intraday.index.max()}")
print(f"Trading days     : {intraday.index.normalize().nunique()}")
print("=" * 70)

intraday.head()

Data source      : yfinance
Rows             : 1,949
Date range       : 2026-07-13 13:30:00+00:00 -> 2026-07-17 19:59:00+00:00
Trading days     : 5


Price,Open,High,Low,Close,Volume
Datetime,,,,,
2026-07-13 13:30:00+00:00,317.0150,319.6299,316.4500,319.5650,1810446
2026-07-13 13:31:00+00:00,319.5500,319.7000,318.6000,319.6696,375556
2026-07-13 13:32:00+00:00,319.6700,319.9100,319.1501,319.2450,513655
2026-07-13 13:33:00+00:00,319.8350,319.8650,319.7800,319.8050,2901
2026-07-13 13:34:00+00:00,319.8050,320.6500,319.8000,320.5964,832242


## 3. Liquidity & Spread Estimation

Market liquidity is a fundamental input to transaction cost analysis because it directly influences execution costs and market impact. Since historical bid and ask quotes are not available through Yahoo Finance, liquidity is estimated from OHLCV data using the Corwin–Schultz (2012) bid-ask spread estimator together with realized volatility and Average Daily Volume (ADV).

The resulting liquidity measures are subsequently used to calibrate the market impact model and construct execution schedules for the trading strategies evaluated throughout this study.

In [ ]:
# =============================================================================
# Liquidity and Spread Estimation
# =============================================================================

def corwin_schultz_spread(df: pd.DataFrame) -> pd.Series:
    """
    Estimate the effective bid-ask spread using the Corwin & Schultz (2012)
    high-low estimator.
    """
    log_hl = (np.log(df["High"]) - np.log(df["Low"])) ** 2
    beta = log_hl + log_hl.shift(1)

    two_bar_high = np.log(df["High"].rolling(2).max())
    two_bar_low = np.log(df["Low"].rolling(2).min())
    gamma = (two_bar_high - two_bar_low) ** 2

    denom = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / denom - np.sqrt(gamma / denom)

    spread = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    return spread.clip(lower=0)


def compute_liquidity_metrics(df: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """
    Compute summary liquidity metrics and append spread/return columns.
    """
    result = df.copy()
    result["log_ret"] = np.log(result["Close"]).diff()
    result["cs_spread"] = corwin_schultz_spread(result)

    n_days = result.index.normalize().nunique()
    adv = result.groupby(result.index.normalize())["Volume"].sum().mean()
    minutes_per_day = int(round(len(result) / n_days))

    sigma_per_min = result["log_ret"].std()
    sigma_annual = sigma_per_min * np.sqrt(252 * minutes_per_day)

    metrics = {
        "n_trading_days": n_days,
        "minutes_per_day": minutes_per_day,
        "ADV_shares": adv,
        "avg_price": result["Close"].mean(),
        "mean_cs_spread_bps": np.nanmean(result["cs_spread"]) * 1e4,
        "median_cs_spread_bps": np.nanmedian(result["cs_spread"]) * 1e4,
        "sigma_per_minute": sigma_per_min,
        "sigma_annualized": sigma_annual,
    }

    return metrics, result


liquidity_metrics, intraday = compute_liquidity_metrics(intraday)

print("=" * 70)
print("Liquidity and Volatility Summary")
print("=" * 70)

for key, value in liquidity_metrics.items():
    if isinstance(value, float):
        print(f"{key:<24}: {value:,.6f}")
    else:
        print(f"{key:<24}: {value}")

print("=" * 70)

ADV = liquidity_metrics["ADV_shares"]
SIGMA_ANNUAL = liquidity_metrics["sigma_annualized"]
AVG_PRICE = liquidity_metrics["avg_price"]
MINUTES_PER_DAY = liquidity_metrics["minutes_per_day"]
HALF_SPREAD = (liquidity_metrics["mean_cs_spread_bps"] / 1e4) / 2

intraday[["Close", "cs_spread"]].tail()

Liquidity and Volatility Summary
n_trading_days          : 5
minutes_per_day         : 390
ADV_shares              : 42,460,274.400000
avg_price               : 324.386346
mean_cs_spread_bps      : 2.663032
median_cs_spread_bps    : 1.521260
sigma_per_minute        : 0.000934
sigma_annualized        : 0.292870


Price,Close,cs_spread
Datetime,,
2026-07-17 19:55:00+00:00,333.8999,0.0006
2026-07-17 19:56:00+00:00,333.7500,0.0014
2026-07-17 19:57:00+00:00,333.9400,0.0009
2026-07-17 19:58:00+00:00,334.1400,0.0008
2026-07-17 19:59:00+00:00,333.7400,0.0010


## 4. Intraday Volume Profile & Liquidity Distribution

The timing of market liquidity plays a central role in institutional trade execution. This section estimates the historical intraday volume profile by calculating the average fraction of daily trading volume executed during each minute of the trading session. The resulting volume distribution serves as the foundation for constructing the VWAP execution schedule and provides a visual representation of how liquidity evolves throughout the trading day.

In [ ]:
# =============================================================================
# Intraday Volume Profile & Liquidity Distribution
# =============================================================================

intraday_local = intraday.copy()

idx = pd.DatetimeIndex(intraday_local.index)
if idx.tz is None:
    idx = idx.tz_localize("UTC")
idx = idx.tz_convert("America/New_York")
intraday_local.index = idx

intraday_local = intraday_local.between_time("09:30", "15:59").copy()

intraday_local["trading_date"] = intraday_local.index.normalize()
intraday_local["minute_of_day"] = (
    intraday_local.index.hour * 60
    + intraday_local.index.minute
    - 9 * 60
    - 30
).astype(int)

intraday_local["minute_of_day"] = intraday_local["minute_of_day"].clip(
    lower=0,
    upper=MINUTES_PER_DAY - 1,
)

volume_curve_raw = intraday_local.groupby("minute_of_day")["Volume"].mean()
volume_curve = (
    volume_curve_raw / volume_curve_raw.sum()
).reindex(range(MINUTES_PER_DAY), fill_value=0.0)

heatmap_pivot = (
    intraday_local.pivot_table(
        index="trading_date",
        columns="minute_of_day",
        values="Volume",
        aggfunc="sum",
    )
    .fillna(0)
    .reindex(columns=range(MINUTES_PER_DAY), fill_value=0)
)

heatmap_log = np.log1p(heatmap_pivot.values)

time_labels = [f"{(9 + (30 + m) // 60):02d}:{(30 + m) % 60:02d}" for m in heatmap_pivot.columns]

fig_heatmap = go.Figure(
    data=go.Heatmap(
        z=heatmap_log,
        x=time_labels,
        y=[d.strftime("%Y-%m-%d") for d in heatmap_pivot.index],
        colorscale="Blues",
        colorbar=dict(title="Log-Scaled Volume"),
        hovertemplate=(
            "Time %{x}<br>"
            "Trading day %{y}<br>"
            "log(1+Volume) %{z:.2f}<extra></extra>"
        ),
    )
)

fig_heatmap.update_layout(
    title=dict(
        text=f"Figure 4.1 | {CONFIG['TICKER']} Intraday Liquidity Distribution",
        x=0.02,
        xanchor="left",
    ),
    template="plotly_white",
    height=460,
    margin=dict(l=70, r=40, t=70, b=50),
    font=dict(family="Arial", size=13),
)

fig_heatmap.update_xaxes(
    title_text="Time of Day (Eastern Time)",
    tickmode="array",
    tickvals=[0, 60, 120, 180, 240, 300, 360],
    ticktext=["09:30", "10:30", "11:30", "12:30", "13:30", "14:30", "15:30"],
)

fig_heatmap.update_yaxes(title_text="Trading day")
fig_heatmap.show()

fig_curve = go.Figure()
fig_curve.add_trace(
    go.Scatter(
        x=volume_curve.index,
        y=volume_curve.values,
        mode="lines",
        fill="tozeroy",
        line=dict(color="#1d4ed8", width=2.5),
        name="Average volume fraction",
        hovertemplate="Minute %{x}<br>Fraction %{y:.4%}<extra></extra>",
    )
)

fig_curve.update_layout(
    title=dict(
        text=f"Figure 4.2 | {CONFIG['TICKER']} Average Intraday Volume Profile",
        x=0.02,
        xanchor="left",
    ),
    xaxis_title="Regular Trading Hours",
    yaxis_title="Fraction of daily volume",
    template="plotly_white",
    height=400,
    margin=dict(l=70, r=40, t=70, b=50),
    font=dict(family="Arial", size=13),
)

fig_curve.update_xaxes(showgrid=True, zeroline=False)
fig_curve.update_yaxes(showgrid=True, zeroline=False)
fig_curve.show()

## 5. Market Impact Model — Almgren–Chriss Framework

Large institutional orders influence market prices while they are being executed. The Almgren–Chriss framework (Almgren & Chriss, 2000; 2001) models this effect by separating execution costs into **temporary** and **permanent** market impact. These components are calibrated using the liquidity estimates obtained in the previous section and provide the foundation for evaluating alternative execution strategies.

### Temporary Market Impact

Temporary market impact represents the execution cost incurred while interacting with available market liquidity. It reflects costs such as crossing the bid–ask spread and consuming resting liquidity from the order book. The impact affects only the current execution and does not permanently alter the market price.

$$
h(v)=\eta v+\epsilon \,\mathrm{sign}(v)
$$

where:

- $\eta$ controls the temporary impact associated with the trading rate.
- $\epsilon$ represents the fixed spread-crossing cost.
- $v$ denotes the trading rate.

---

### Permanent Market Impact

Permanent market impact represents the lasting price change created by trading activity. Large institutional orders reveal information to the market, causing subsequent prices to adjust.

$$
g(v)=\gamma v
$$

where $\gamma$ is the permanent impact coefficient.

The impact coefficients ($\eta$, $\gamma$) are calibrated using the estimated volatility and Average Daily Volume (ADV), while the spread component ($\epsilon$) is estimated from the Corwin–Schultz bid–ask spread model.

---

### Optimal Execution

The objective of the Almgren–Chriss framework is to determine an execution schedule that minimizes expected implementation shortfall while controlling execution risk. The optimal inventory trajectory is given by

$$
x_j
=
X
\cdot
\frac{\sinh(\kappa(T-t_j))}
{\sinh(\kappa T)},
\qquad
\kappa
=
\frac{1}{\tau}
\cosh^{-1}
\left(
\frac{\tilde{\kappa}^{2}\tau^{2}}{2}
+
1
\right),
\qquad
\tilde{\kappa}^{2}
=
\frac{\lambda\sigma^{2}}
{\tilde{\eta}}
$$

where

$$
\tilde{\eta}
=
\eta
-
\frac{1}{2}
\gamma\tau
$$

Higher values of the risk-aversion parameter ($\lambda$) produce more aggressive execution schedules that reduce exposure to uncertain price movements while accepting greater market impact. Lower values of $\lambda$ spread the order more evenly across the trading horizon, reducing market impact at the expense of increased timing risk.

---

**Key Takeaway**

The Almgren–Chriss framework provides a quantitative benchmark for balancing **market impact** and **execution risk**. Although modern execution systems incorporate richer market microstructure signals and adaptive algorithms, this model remains one of the most widely used foundations for transaction cost analysis and optimal execution research.

In [ ]:
# =============================================================================
# Calibrate Almgren–Chriss Impact Parameters
# =============================================================================

def calibrate_impact_parameters(
    adv: float,
    sigma_annual: float,
    avg_price: float,
    half_spread_frac: float,
    minutes_per_day: int = 390,
) -> dict:
    """
    Calibrate Almgren–Chriss impact parameters from observed liquidity inputs.
    """
    sigma_daily = sigma_annual / np.sqrt(252)
    sigma_per_min = sigma_daily / np.sqrt(minutes_per_day)

    ref_participation = 0.01
    sqrt_law_impact_frac = 0.5 * sigma_daily * np.sqrt(ref_participation)
    ref_shares = ref_participation * adv

    eta = (sqrt_law_impact_frac * avg_price) / ref_shares
    gamma_perm = 0.3 * eta
    epsilon = half_spread_frac * avg_price

    return {
        "eta": eta,
        "gamma_perm": gamma_perm,
        "epsilon": epsilon,
        "sigma_per_min": sigma_per_min,
        "sigma_daily": sigma_daily,
    }


impact_params = calibrate_impact_parameters(
    adv=ADV,
    sigma_annual=SIGMA_ANNUAL,
    avg_price=AVG_PRICE,
    half_spread_frac=HALF_SPREAD,
    minutes_per_day=MINUTES_PER_DAY,
)

print("=" * 70)
print("Calibrated Almgren-Chriss Impact Parameters")
print("=" * 70)

for key, value in impact_params.items():
    print(f"{key:<16}: {value:.8f}")

pct_of_adv = CONFIG["ORDER_SIZE_SHARES"] / ADV * 100
print("=" * 70)
print(
    f"Parent order = {CONFIG['ORDER_SIZE_SHARES']:,} shares "
    f"({pct_of_adv:.2f}% of ADV of {ADV:,.0f} shares)"
)
print("=" * 70)

Calibrated Almgren-Chriss Impact Parameters
eta             : 0.00000070
gamma_perm      : 0.00000021
epsilon         : 0.04319256
sigma_per_min   : 0.00093420
sigma_daily     : 0.01844906
Parent order = 250,000 shares (0.59% of ADV of 42,460,274 shares)


## 6. Execution Strategy Schedules

Execution quality depends not only on the size of an order but also on how that order is distributed throughout the trading session. This section constructs four widely used execution schedules that represent different approaches to balancing market impact, liquidity, and execution risk. Each strategy generates a sequence of trades that will be evaluated under identical market conditions in the simulation framework.

| Strategy | Execution Logic |
|-----------|-----------------|
| **TWAP** | Executes an equal quantity of shares during each execution interval. |
| **VWAP** | Allocates shares according to the historical intraday volume profile, concentrating execution during periods of higher market liquidity. |
| **POV (Percentage of Volume)** | Maintains a fixed participation rate relative to projected market volume throughout the trading session. |
| **IS-Optimal (Almgren–Chriss)** | Computes a risk-adjusted execution schedule that minimizes the trade-off between expected transaction costs and execution risk. |

In [ ]:
# =============================================================================
# Build Execution Schedules
# =============================================================================

def resample_curve_to_slices(curve: pd.Series, n_slices: int) -> np.ndarray:
    """Aggregate a minute-level volume curve into execution slices."""
    minutes = len(curve)
    bin_edges = np.linspace(0, minutes, n_slices + 1).astype(int)

    slice_frac = np.array(
        [curve.iloc[bin_edges[i]:bin_edges[i + 1]].sum() for i in range(n_slices)]
    )

    return slice_frac / slice_frac.sum()


def twap_schedule(order_size: float, n_slices: int) -> np.ndarray:
    return np.full(n_slices, order_size / n_slices)


def vwap_schedule(order_size: float, slice_volume_curve: np.ndarray) -> np.ndarray:
    return order_size * slice_volume_curve


def pov_schedule(
    order_size: float,
    slice_volume_curve: np.ndarray,
    adv: float,
    target_rate: float,
) -> np.ndarray:
    projected_slice_volume = slice_volume_curve * adv
    raw_alloc = projected_slice_volume * target_rate
    return raw_alloc / raw_alloc.sum() * order_size


def almgren_chriss_schedule(
    order_size: float,
    n_slices: int,
    sigma_per_slice: float,
    risk_aversion: float,
    eta: float,
    gamma_perm: float,
    horizon: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    tau = horizon / n_slices
    eta_tilde = eta - 0.5 * gamma_perm * tau
    eta_tilde = eta_tilde if eta_tilde > 1e-12 else eta * 0.5

    kappa_tilde_sq = max((risk_aversion * sigma_per_slice**2) / eta_tilde, 1e-14)
    kappa = np.arccosh(0.5 * kappa_tilde_sq * tau**2 + 1) / tau

    times = np.linspace(0, horizon, n_slices + 1)

    if kappa * horizon > 1e-8:
        holdings = order_size * np.sinh(kappa * (horizon - times)) / np.sinh(kappa * horizon)
    else:
        holdings = order_size * (1 - times / horizon)

    trades = -np.diff(holdings)
    return trades, holdings


slice_volume_curve = resample_curve_to_slices(volume_curve, CONFIG["N_SLICES"])
sigma_per_slice = impact_params["sigma_per_min"] * np.sqrt(MINUTES_PER_DAY / CONFIG["N_SLICES"])

ac_trades, ac_holdings = almgren_chriss_schedule(
    order_size=CONFIG["ORDER_SIZE_SHARES"],
    n_slices=CONFIG["N_SLICES"],
    sigma_per_slice=sigma_per_slice,
    risk_aversion=CONFIG["RISK_AVERSION"],
    eta=impact_params["eta"],
    gamma_perm=impact_params["gamma_perm"],
)

schedules = {
    "TWAP": twap_schedule(CONFIG["ORDER_SIZE_SHARES"], CONFIG["N_SLICES"]),
    "VWAP": vwap_schedule(CONFIG["ORDER_SIZE_SHARES"], slice_volume_curve),
    "POV": pov_schedule(
        CONFIG["ORDER_SIZE_SHARES"],
        slice_volume_curve,
        ADV,
        CONFIG["PARTICIPATION_TARGET"],
    ),
    "IS-Optimal (AC)": ac_trades,
}

schedule_df = pd.DataFrame(
    schedules,
    index=[f"Slice {i+1}" for i in range(CONFIG["N_SLICES"])],
)
schedule_df.loc["TOTAL"] = schedule_df.sum()
schedule_df.round(0)

,TWAP,VWAP,POV,IS-Optimal (AC)
Slice 1,"19,231.0000","48,722.0000","48,722.0000","25,159.0000"
Slice 2,"19,231.0000","24,264.0000","24,264.0000","23,659.0000"
Slice 3,"19,231.0000","18,392.0000","18,392.0000","22,317.0000"
Slice 4,"19,231.0000","13,040.0000","13,040.0000","21,123.0000"
Slice 5,"19,231.0000","14,488.0000","14,488.0000","20,071.0000"
Slice 6,"19,231.0000","15,023.0000","15,023.0000","19,152.0000"
Slice 7,"19,231.0000","14,776.0000","14,776.0000","18,362.0000"
Slice 8,"19,231.0000","13,700.0000","13,700.0000","17,693.0000"
Slice 9,"19,231.0000","11,812.0000","11,812.0000","17,143.0000"
Slice 10,"19,231.0000","11,497.0000","11,497.0000","16,707.0000"


## 7. Monte Carlo Execution Simulation

Execution costs are inherently uncertain because market prices evolve continuously while an order is being executed. A single deterministic simulation cannot capture this uncertainty. To evaluate the robustness of each execution strategy, this study generates **`N_MC_PATHS` independent stochastic price paths** using a Monte Carlo simulation.

For each simulated path:

- the unaffected market price evolves according to the calibrated volatility,
- temporary market impact affects the execution price of each trade,
- permanent market impact shifts future prices following execution,
- implementation shortfall is measured relative to the arrival price.

Repeating this process across all simulated price paths produces a distribution of execution outcomes rather than a single estimate, allowing both expected transaction costs and execution risk to be compared across execution strategies.

In [ ]:
# =============================================================================
# Monte Carlo Execution Simulation
# =============================================================================

def simulate_strategy(
    trade_schedule: np.ndarray,
    arrival_price: float,
    sigma_per_slice: float,
    eta: float,
    gamma_perm: float,
    epsilon: float,
    side: str = "BUY",
    n_paths: int = 2000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Simulate implementation shortfall for a fixed execution schedule.
    """
    rng = np.random.default_rng(seed)
    sign = 1 if side.upper() == "BUY" else -1
    n_slices = len(trade_schedule)

    total_shortfall = np.zeros(n_paths)
    spread_cost = np.zeros(n_paths)
    temp_impact_cost = np.zeros(n_paths)
    perm_impact_cost = np.zeros(n_paths)
    timing_cost = np.zeros(n_paths)

    shocks = rng.normal(0, sigma_per_slice, size=(n_paths, n_slices))

    for path_idx in range(n_paths):
        mid = arrival_price

        for slice_idx in range(n_slices):
            trade_size = trade_schedule[slice_idx]
            mid_before = mid

            mid = mid * (1 + shocks[path_idx, slice_idx])
            temp = eta * trade_size
            perm = gamma_perm * trade_size

            exec_price = mid + sign * (temp + epsilon)
            mid = mid + sign * perm

            spread_cost[path_idx] += sign * trade_size * epsilon
            temp_impact_cost[path_idx] += sign * trade_size * temp
            perm_impact_cost[path_idx] += sign * trade_size * perm
            timing_cost[path_idx] += sign * trade_size * (mid_before * shocks[path_idx, slice_idx])
            total_shortfall[path_idx] += trade_size * sign * (exec_price - arrival_price)

    notional = arrival_price * trade_schedule.sum()

    results = pd.DataFrame(
        {
            "total_shortfall_$": total_shortfall,
            "spread_cost_$": spread_cost,
            "temp_impact_$": temp_impact_cost,
            "perm_impact_$": perm_impact_cost,
            "timing_cost_$": timing_cost,
        }
    )

    results_bps = results / notional * 1e4
    results_bps.columns = [col.replace("_$", "_bps") for col in results.columns]

    return pd.concat([results, results_bps], axis=1)


def simulate_pov_strategy(
    order_size: float,
    slice_volume_curve: np.ndarray,
    adv: float,
    target_rate: float,
    arrival_price: float,
    sigma_per_slice: float,
    eta: float,
    gamma_perm: float,
    epsilon: float,
    side: str = "BUY",
    volume_noise_std: float = 0.35,
    n_paths: int = 2000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Simulate a reactive POV strategy using stochastic realized volume.
    """
    rng = np.random.default_rng(seed)
    sign = 1 if side.upper() == "BUY" else -1
    n_slices = len(slice_volume_curve)
    projected_slice_volume = slice_volume_curve * adv

    total_shortfall = np.zeros(n_paths)
    spread_cost = np.zeros(n_paths)
    temp_impact_cost = np.zeros(n_paths)
    perm_impact_cost = np.zeros(n_paths)
    timing_cost = np.zeros(n_paths)
    fill_ratio_at_close = np.zeros(n_paths)

    price_shocks = rng.normal(0, sigma_per_slice, size=(n_paths, n_slices))
    volume_shocks = rng.lognormal(
        mean=-0.5 * volume_noise_std**2,
        sigma=volume_noise_std,
        size=(n_paths, n_slices),
    )

    for path_idx in range(n_paths):
        mid = arrival_price
        remaining = order_size

        for slice_idx in range(n_slices):
            realized_volume = projected_slice_volume[slice_idx] * volume_shocks[path_idx, slice_idx]
            trade_size = min(target_rate * realized_volume, remaining)

            if slice_idx == n_slices - 1:
                trade_size = remaining

            remaining -= trade_size

            mid_before = mid
            mid = mid * (1 + price_shocks[path_idx, slice_idx])

            temp = eta * trade_size
            perm = gamma_perm * trade_size

            exec_price = mid + sign * (temp + epsilon)
            mid = mid + sign * perm

            spread_cost[path_idx] += sign * trade_size * epsilon
            temp_impact_cost[path_idx] += sign * trade_size * temp
            perm_impact_cost[path_idx] += sign * trade_size * perm
            timing_cost[path_idx] += sign * trade_size * (mid_before * price_shocks[path_idx, slice_idx])
            total_shortfall[path_idx] += trade_size * sign * (exec_price - arrival_price)

        fill_ratio_at_close[path_idx] = 1 - max(remaining, 0) / order_size

    notional = arrival_price * order_size

    results = pd.DataFrame(
        {
            "total_shortfall_$": total_shortfall,
            "spread_cost_$": spread_cost,
            "temp_impact_$": temp_impact_cost,
            "perm_impact_$": perm_impact_cost,
            "timing_cost_$": timing_cost,
        }
    )

    results_bps = results / notional * 1e4
    results_bps.columns = [col.replace("_$", "_bps") for col in results.columns]

    out = pd.concat([results, results_bps], axis=1)
    out["fill_ratio_at_close"] = fill_ratio_at_close
    return out


mc_results = {}
for name, sched in schedules.items():
    if name == "POV":
        continue
    mc_results[name] = simulate_strategy(
        trade_schedule=sched,
        arrival_price=AVG_PRICE,
        sigma_per_slice=sigma_per_slice,
        eta=impact_params["eta"],
        gamma_perm=impact_params["gamma_perm"],
        epsilon=impact_params["epsilon"],
        side=CONFIG["ORDER_SIDE"],
        n_paths=CONFIG["N_MC_PATHS"],
        seed=CONFIG["RANDOM_SEED"],
    )

mc_results["POV"] = simulate_pov_strategy(
    order_size=CONFIG["ORDER_SIZE_SHARES"],
    slice_volume_curve=slice_volume_curve,
    adv=ADV,
    target_rate=CONFIG["PARTICIPATION_TARGET"],
    arrival_price=AVG_PRICE,
    sigma_per_slice=sigma_per_slice,
    eta=impact_params["eta"],
    gamma_perm=impact_params["gamma_perm"],
    epsilon=impact_params["epsilon"],
    side=CONFIG["ORDER_SIDE"],
    n_paths=CONFIG["N_MC_PATHS"],
    seed=CONFIG["RANDOM_SEED"],
)

print(f"Simulated {CONFIG['N_MC_PATHS']:,} price paths for each of {len(schedules)} strategies.")
print(
    f"Mean POV fill ratio before final-slice catch-up: "
    f"{mc_results['POV']['fill_ratio_at_close'].mean():.2%} (1.00 = fully organic fill)"
)

mc_results["TWAP"].head()

Simulated 2,000 price paths for each of 4 strategies.
Mean POV fill ratio before final-slice catch-up: 100.00% (1.00 = fully organic fill)


,total_shortfall_$,spread_cost_$,temp_impact_$,perm_impact_$,timing_cost_$,total_shortfall_bps,spread_cost_bps,temp_impact_bps,perm_impact_bps,timing_cost_bps
0,"-591,509.1353","10,798.1402","3,388.1350","1,016.4405","-52,733.6487",-72.9388,1.3315,0.4178,0.1253,-6.5026
1,"376,149.7289","10,798.1402","3,388.1350","1,016.4405","12,116.5846",46.3829,1.3315,0.4178,0.1253,1.4941
2,"1,058,811.3005","10,798.1402","3,388.1350","1,016.4405","67,154.6481",130.5618,1.3315,0.4178,0.1253,8.2808
3,"1,038,504.2658","10,798.1402","3,388.1350","1,016.4405","148,316.4147",128.0577,1.3315,0.4178,0.1253,18.2889
4,"-1,016,232.0029","10,798.1402","3,388.1350","1,016.4405","-68,209.6019",-125.3113,1.3315,0.4178,0.1253,-8.4109


## 8. Execution Quality Comparison

A successful execution strategy should minimize transaction costs while remaining consistent across different market conditions. This section compares the performance of each execution algorithm using the simulated execution results.

The comparison focuses on three commonly used Transaction Cost Analysis (TCA) metrics:

- **Mean Implementation Shortfall** – average execution cost relative to the arrival price.
- **Execution Risk** – variability of implementation shortfall across simulated price paths.
- **95th Percentile Cost** – execution cost under adverse market conditions.

Together, these metrics provide a practical view of the trade-off between execution cost and execution risk.

In [ ]:
# =============================================================================
# Execution Performance Summary
# =============================================================================

def summarize_strategy(strategy_name: str, results: pd.DataFrame) -> dict:
    shortfall = results["total_shortfall_bps"]

    return {
        "Strategy": strategy_name,
        "Mean IS (bps)": shortfall.mean(),
        "Execution Risk (bps)": shortfall.std(),
        "Median IS (bps)": shortfall.median(),
        "95th %ile (bps)": shortfall.quantile(0.95),
        "5th %ile (bps)": shortfall.quantile(0.05),
        "Cost > 0 (%)": (shortfall > 0).mean() * 100,
        "Spread (bps)": results["spread_cost_bps"].mean(),
        "Temporary Impact (bps)": results["temp_impact_bps"].mean(),
        "Permanent Impact (bps)": results["perm_impact_bps"].mean(),
        "Timing Risk (bps)": results["timing_cost_bps"].std(),
    }


summary_table = pd.DataFrame(
    [summarize_strategy(strategy, result) for strategy, result in mc_results.items()]
).sort_values("Mean IS (bps)").reset_index(drop=True)

summary_table["Rank"] = np.arange(1, len(summary_table) + 1)

ranking_table = summary_table[
    ["Rank", "Strategy", "Mean IS (bps)", "Execution Risk (bps)", "95th %ile (bps)"]
].copy()

cost_breakdown = summary_table[
    ["Strategy", "Spread (bps)", "Temporary Impact (bps)", "Permanent Impact (bps)", "Timing Risk (bps)"]
].copy()

ranking_styled = (
    ranking_table.style
    .hide(axis="index")
    .format(
        {
            "Rank": "{:.0f}",
            "Mean IS (bps)": "{:.2f}",
            "Execution Risk (bps)": "{:.2f}",
            "95th %ile (bps)": "{:.2f}",
        }
    )
    .set_properties(subset=["Strategy"], **{"text-align": "left"})
    .set_properties(
        subset=["Rank", "Mean IS (bps)", "Execution Risk (bps)", "95th %ile (bps)"],
        **{"text-align": "right"},
    )
    .set_table_styles(
        [
            {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("padding", "6px 10px")]},
        ]
    )
)

cost_styled = (
    cost_breakdown.style
    .hide(axis="index")
    .format(
        {
            "Spread (bps)": "{:.2f}",
            "Temporary Impact (bps)": "{:.2f}",
            "Permanent Impact (bps)": "{:.2f}",
            "Timing Risk (bps)": "{:.2f}",
        }
    )
    .set_properties(subset=["Strategy"], **{"text-align": "left"})
    .set_properties(
        subset=["Spread (bps)", "Temporary Impact (bps)", "Permanent Impact (bps)", "Timing Risk (bps)"],
        **{"text-align": "right"},
    )
    .set_table_styles(
        [
            {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("padding", "6px 10px")]},
        ]
    )
)

print("Execution Strategy Ranking")
display(ranking_styled)

print()
print("Execution Cost Decomposition")
display(cost_styled)

best_strategy = ranking_table.iloc[0]

print()
print("Key Insight")
print(
    f"{best_strategy['Strategy']} achieved the lowest average implementation shortfall "
    f"({best_strategy['Mean IS (bps)']:.2f} bps) among the {len(summary_table)} strategies evaluated."
)

Execution Strategy Ranking


Rank,Strategy,Mean IS (bps),Execution Risk (bps),95th %ile (bps)
1,IS-Optimal (AC),3.58,107.43,179.12
2,TWAP,3.89,114.26,190.86
3,VWAP,3.93,102.11,174.02
4,POV,5.96,51.32,89.40



Execution Cost Decomposition


Strategy,Spread (bps),Temporary Impact (bps),Permanent Impact (bps),Timing Risk (bps)
IS-Optimal (AC),1.33,0.43,0.13,14.51
TWAP,1.33,0.42,0.13,14.33
VWAP,1.33,0.54,0.16,16.33
POV,1.33,5.43,1.63,51.32



Key Insight
IS-Optimal (AC) achieved the lowest average implementation shortfall (3.58 bps) among the 4 strategies evaluated.


### Interpretation

The execution summary compares each strategy using three key Transaction Cost Analysis (TCA) metrics:

- **Mean IS (bps)** – Average implementation shortfall relative to the arrival price. Lower values indicate lower execution costs.
- **Execution Risk (bps)** – Standard deviation of implementation shortfall across all simulated price paths. Lower values indicate more consistent execution.
- **95th Percentile (bps)** – Execution cost under adverse market conditions. Lower values indicate better downside protection.

The **IS-Optimal (Almgren–Chriss)** strategy is designed to balance execution cost and execution risk. Depending on the selected risk-aversion parameter, it may accept slightly higher expected costs in exchange for more stable execution outcomes.

## 9. Visualizations

### 9.1 Execution Trajectories

Each execution strategy follows a different trading schedule throughout the execution horizon. The chart below compares the cumulative percentage of the parent order executed over time, illustrating how aggressively or conservatively each algorithm distributes trades across the trading session.


In [ ]:
# =============================================================================
# Execution Trajectories
# =============================================================================

fig = go.Figure()

colors = {
    "TWAP": "#2563EB",
    "VWAP": "#0F766E",
    "POV": "#EA580C",
    "IS-Optimal (AC)": "#7C3AED",
}

for strategy, schedule in schedules.items():

    cumulative_execution = np.concatenate(([0], np.cumsum(schedule)))
    execution_pct = cumulative_execution / cumulative_execution[-1] * 100
    execution_step = np.arange(len(cumulative_execution))

    fig.add_trace(
        go.Scatter(
            x=execution_step,
            y=execution_pct,
            mode="lines+markers",
            name=strategy,
            line=dict(width=3, color=colors[strategy]),
            marker=dict(size=7),
        )
    )

fig.update_layout(
    title=dict(
        text="Figure 9.1 | Execution Trajectory Comparison",
        x=0.02,
        xanchor="left",
    ),
    xaxis_title="Execution Interval",
    yaxis_title="Parent Order Completed (%)",
    template="plotly_white",
    height=500,
    width=950,
    legend_title="Execution Strategy",
    hovermode="x unified",
    margin=dict(l=70, r=40, t=70, b=50),
)

fig.update_yaxes(range=[0, 100])

fig.show()

### 9.2 Slippage Distributions

This chart compares the full distribution of simulated implementation shortfall for each execution strategy. It shows not only average cost, but also how much execution outcomes vary across market scenarios, which is important when comparing execution risk.


In [ ]:
# =============================================================================
# Slippage Distributions
# =============================================================================

STRATEGY_COLORS = {
    "TWAP": "#2563EB",            # Blue
    "VWAP": "#059669",            # Green
    "POV": "#EA580C",             # Orange
    "IS-Optimal (AC)": "#DC2626", # Red
}

violin_data = pd.concat(
    [
        pd.DataFrame(
            {
                "Strategy": strategy_name,
                "Implementation Shortfall (bps)": result["total_shortfall_bps"],
            }
        )
        for strategy_name, result in mc_results.items()
    ],
    ignore_index=True,
)

fig_violin = px.violin(
    violin_data,
    x="Strategy",
    y="Implementation Shortfall (bps)",
    color="Strategy",
    color_discrete_map=STRATEGY_COLORS,
    box=True,
    points=False,
    title=f"Figure 9.2 | Implementation Shortfall Distribution ({CONFIG['N_MC_PATHS']:,} Monte Carlo Paths)",
)

fig_violin.update_traces(
    width=0.65,
    opacity=0.85,
    meanline_visible=True,
)

fig_violin.update_layout(
    template="plotly_white",
    height=460,
    width=950,
    showlegend=False,
    margin=dict(l=70, r=40, t=70, b=50),
    title=dict(x=0.02, xanchor="left"),
    xaxis_title="Execution Strategy",
    yaxis_title="Implementation Shortfall (bps)",
    violingap=0.15,
    violinmode="group",
    hovermode="x unified",
)

fig_violin.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    line_width=1.5,
    opacity=0.6,
)

fig_violin.show()

### 9.3 Cost Decomposition

This chart breaks average execution cost into spread cost, temporary impact, permanent impact, and timing risk. It helps explain not only which strategy was cheapest on average, but also where the cost came from.


In [ ]:
# =============================================================================
# Cost Decomposition
# =============================================================================

plot_df = summary_table.sort_values("Rank").set_index("Strategy")

components = ["Spread (bps)", "Temporary Impact (bps)", "Permanent Impact (bps)"]
component_colors = ["#f59e0b", "#dc2626", "#7c3aed"]

fig_decomp = go.Figure()

for component, color in zip(components, component_colors):
    fig_decomp.add_trace(
        go.Bar(
            x=plot_df.index,
            y=plot_df[component],
            name=component,
            marker_color=color,
        )
    )

fig_decomp.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df["Timing Risk (bps)"],
        name="Timing Risk (bps)",
        mode="lines+markers",
        marker=dict(size=10, symbol="diamond", color="#111827"),
        line=dict(color="#111827", dash="dot", width=2),
        yaxis="y2",
    )
)

fig_decomp.update_layout(
    barmode="stack",
    title=dict(
        text="Figure 9.3 | Execution Cost Decomposition",
        x=0.02,
        xanchor="left",
    ),
    xaxis_title="Strategy",
    yaxis=dict(title="Average cost (bps)"),
    yaxis2=dict(
        title="Timing risk (bps)",
        overlaying="y",
        side="right",
        showgrid=False,
    ),
    template="plotly_white",
    height=480,
    width=950,
    legend=dict(orientation="h", y=-0.25),
    margin=dict(l=70, r=60, t=70, b=80),
    hovermode="x unified",
)

fig_decomp.show()

### 9.4 Cost vs. Risk Efficient Frontier

This chart compares average execution cost with execution risk across strategies. It makes the main trade-off visible: strategies that trade more aggressively can reduce risk but often increase expected cost.


In [ ]:
# =============================================================================
# Cost vs. Risk Frontier
# =============================================================================

fig_frontier = go.Figure()

fig_frontier.add_trace(
    go.Scatter(
        x=summary_table["Execution Risk (bps)"],
        y=summary_table["Mean IS (bps)"],
        mode="markers+text",
        text=summary_table["Strategy"],
        textposition="top center",
        marker=dict(
            size=16,
            color=[colors.get(strategy, "#333333") for strategy in summary_table["Strategy"]],
            line=dict(width=1, color="white"),
        ),
    )
)

fig_frontier.update_layout(
    title=dict(
        text="Figure 9.4 | Cost vs. Risk Frontier",
        x=0.02,
        xanchor="left",
    ),
    xaxis_title="Execution Risk (bps)",
    yaxis_title="Mean Implementation Shortfall (bps)",
    template="plotly_white",
    height=460,
    width=950,
    margin=dict(l=70, r=40, t=70, b=60),
)

fig_frontier.show()

## 10. Sensitivity Analysis — Risk Aversion

The Almgren–Chriss model allows the execution schedule to be adjusted through the risk-aversion parameter (λ). This section examines how different values of λ affect the optimal execution trajectory.

Higher values of λ prioritize reducing exposure to market risk by executing more aggressively, while lower values spread trades more evenly across the trading horizon to reduce market impact.


In [ ]:
# =============================================================================
# Risk Aversion Sensitivity Analysis
# =============================================================================

lambda_grid = [1e-8, 1e-7, 1e-6, 5e-6, 2e-5, 1e-4]

fig = go.Figure()

palette = px.colors.sequential.Blues

for idx, lam in enumerate(lambda_grid):

    trades, holdings = almgren_chriss_schedule(
        order_size=CONFIG["ORDER_SIZE_SHARES"],
        n_slices=CONFIG["N_SLICES"],
        sigma_per_slice=sigma_per_slice,
        risk_aversion=lam,
        eta=impact_params["eta"],
        gamma_perm=impact_params["gamma_perm"],
    )

    pct_remaining = holdings / holdings[0] * 100

    fig.add_trace(
        go.Scatter(
            x=np.arange(len(holdings)),
            y=pct_remaining,
            mode="lines+markers",
            name=f"λ = {lam:.0e}",
            line=dict(
                width=3,
                color=palette[int(idx / (len(lambda_grid) - 1) * (len(palette) - 1))]
            ),
        )
    )

fig.update_layout(
    title=dict(
        text="Figure 10.1 | Optimal Execution Under Different Risk Aversion Levels",
        x=0.02,
        xanchor="left",
    ),
    xaxis_title="Execution Interval",
    yaxis_title="Parent Order Remaining (%)",
    template="plotly_white",
    height=460,
    width=950,
    margin=dict(l=70, r=40, t=70, b=60),
    legend_title="Risk Aversion (λ)",
)

fig.show()

## 11. Order-Size Scenario Analysis

This section stress-tests the execution strategies across a range of parent order sizes expressed as a percentage of ADV. It shows how expected implementation shortfall changes as the order becomes larger relative to market liquidity.

In [ ]:
# =============================================================================
# Order-Size Scenario Analysis
# =============================================================================

pct_adv_scenarios = [0.01, 0.02, 0.05, 0.10, 0.20, 0.35]
scenario_rows = []

strategy_colors = {
    "TWAP": "#2563EB",
    "VWAP": "#059669",
    "IS-Optimal (AC)": "#DC2626",
}

for pct_adv in pct_adv_scenarios:
    order_size = pct_adv * ADV

    ac_schedule, _ = almgren_chriss_schedule(
        order_size=order_size,
        n_slices=CONFIG["N_SLICES"],
        sigma_per_slice=sigma_per_slice,
        risk_aversion=CONFIG["RISK_AVERSION"],
        eta=impact_params["eta"],
        gamma_perm=impact_params["gamma_perm"],
    )

    scenario_schedules = {
        "TWAP": twap_schedule(order_size, CONFIG["N_SLICES"]),
        "VWAP": vwap_schedule(order_size, slice_volume_curve),
        "IS-Optimal (AC)": ac_schedule,
    }

    for strategy_name, schedule in scenario_schedules.items():
        results = simulate_strategy(
            trade_schedule=schedule,
            arrival_price=AVG_PRICE,
            sigma_per_slice=sigma_per_slice,
            eta=impact_params["eta"],
            gamma_perm=impact_params["gamma_perm"],
            epsilon=impact_params["epsilon"],
            side=CONFIG["ORDER_SIDE"],
            n_paths=500,
            seed=1,
        )

        scenario_rows.append(
            {
                "% of ADV": pct_adv * 100,
                "Strategy": strategy_name,
                "Mean IS (bps)": results["total_shortfall_bps"].mean(),
            }
        )

scenario_df = pd.DataFrame(scenario_rows).sort_values(["Strategy", "% of ADV"])

fig_scenario = go.Figure()

for strategy_name in ["TWAP", "VWAP", "IS-Optimal (AC)"]:
    subset = scenario_df[scenario_df["Strategy"] == strategy_name].sort_values("% of ADV")

    fig_scenario.add_trace(
        go.Scatter(
            x=subset["% of ADV"],
            y=subset["Mean IS (bps)"],
            mode="lines+markers",
            name=strategy_name,
            line=dict(
                color=strategy_colors[strategy_name],
                width=3.5 if strategy_name == "IS-Optimal (AC)" else 2.5,
            ),
            marker=dict(
                size=8,
                color=strategy_colors[strategy_name],
            ),
        )
    )

fig_scenario.update_layout(
    title=dict(
        text="Figure 11.1 | Expected Implementation Shortfall vs. Order Size",
        x=0.02,
        xanchor="left",
    ),
    template="plotly_white",
    height=460,
    width=950,
    xaxis_title="Order Size (% of ADV)",
    yaxis_title="Expected Implementation Shortfall (bps)",
    margin=dict(l=70, r=40, t=70, b=50),
    legend_title="Strategy",
    hovermode="x unified",
)

fig_scenario.update_xaxes(showgrid=True, zeroline=False)
fig_scenario.update_yaxes(showgrid=True, zeroline=False)

fig_scenario.show()

# 12. Project Summary

This project implemented a Transaction Cost Analysis (TCA) framework for comparing institutional execution strategies using intraday equity market data.

The workflow included:

- Estimating market liquidity using the Corwin–Schultz bid–ask spread estimator.
- Calibrating an Almgren–Chriss market impact model.
- Simulating TWAP, VWAP, POV, and IS-Optimal execution strategies.
- Evaluating implementation shortfall, execution risk, and transaction cost decomposition using Monte Carlo simulation.
- Comparing execution performance across different order sizes and risk-aversion settings.

## Key Findings

- Execution quality depends on both market liquidity and execution schedule.
- Lower implementation shortfall does not always imply lower execution risk.
- Increasing order size leads to higher expected execution costs.
- Higher risk aversion produces more aggressive execution schedules that reduce exposure to price uncertainty while increasing market impact.

## Future Improvements

Potential extensions include:

- Real-time market data integration
- Limit order book simulation (LOBSTER)
- Multi-venue execution modeling
- Reinforcement learning for adaptive execution
- Interactive Streamlit dashboard